In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

bronze = spark.table("workspace.default.bronze_weather")
print(bronze.count())
bronze.select("area", "time", "data").show(1, truncate=False)


5139
+----+--------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|area|time                |data                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
+----+---------

In [0]:
weather = bronze.select(
    "area",
    F.to_timestamp("time").alias("time_utc"),
    F.get_json_object("data", "$.instant.details.air_temperature").cast("double").alias("temperature_c"),
    F.get_json_object("data", "$.instant.details.wind_speed").cast("double").alias("wind_speed_ms"),
    F.get_json_object("data", "$.next_1_hours.details.precipitation_amount").cast("double").alias("precip_1h_mm"),
    "source_file",
)
weather.show(3, truncate=False)

+----+-------------------+-------------+-------------+------------+----------------------------------------------+
|area|time_utc           |temperature_c|wind_speed_ms|precip_1h_mm|source_file                                   |
+----+-------------------+-------------+-------------+------------+----------------------------------------------+
|NO1 |2026-10-09 18:00:00|8.6          |2.4          |0.0         |weather/2026-10-09/weather_2026-10-09_NO1.json|
|NO1 |2026-10-09 19:00:00|8.4          |2.6          |0.0         |weather/2026-10-09/weather_2026-10-09_NO1.json|
|NO1 |2026-10-09 20:00:00|8.0          |3.1          |0.0         |weather/2026-10-09/weather_2026-10-09_NO1.json|
+----+-------------------+-------------+-------------+------------+----------------------------------------------+
only showing top 3 rows


In [0]:
print(weather.count())
print(weather.select("area", "time_utc").distinct().count())
print(weather.filter(F.col("precip_1h_mm").isNull()).count())

5139
1715
1882


In [0]:
silver = (
    weather
    .withColumn("forecast_date", F.to_date(F.substring("source_file", 9, 10)))
    .dropDuplicates(["area", "time_utc", "forecast_date"])
)

print(silver.count())

5139


In [0]:
silver.write.mode("overwrite").saveAsTable("workspace.default.silver_weather")
spark.table("workspace.default.silver_weather").orderBy("area", "time_utc", "forecast_date").show(5, truncate=False)

+----+-------------------+-------------+-------------+------------+----------------------------------------------+-------------+
|area|time_utc           |temperature_c|wind_speed_ms|precip_1h_mm|source_file                                   |forecast_date|
+----+-------------------+-------------+-------------+------------+----------------------------------------------+-------------+
|NO1 |2026-10-09 18:00:00|8.6          |2.4          |0.0         |weather/2026-10-09/weather_2026-10-09_NO1.json|2026-10-09   |
|NO1 |2026-10-09 19:00:00|8.4          |2.6          |0.0         |weather/2026-10-09/weather_2026-10-09_NO1.json|2026-10-09   |
|NO1 |2026-10-09 20:00:00|8.0          |3.1          |0.0         |weather/2026-10-09/weather_2026-10-09_NO1.json|2026-10-09   |
|NO1 |2026-10-09 21:00:00|7.8          |3.2          |0.0         |weather/2026-10-09/weather_2026-10-09_NO1.json|2026-10-09   |
|NO1 |2026-10-09 22:00:00|7.7          |4.0          |0.0         |weather/2026-10-09/weather_202

In [0]:
(
    spark.table("workspace.default.silver_weather")
    .filter((F.col("area") == "NO5") & (F.col("time_utc") == "2026-10-22 12:00:00"))
    .select("forecast_date", "temperature_c", "wind_speed_ms", "precip_1h_mm")
    .orderBy("forecast_date")
    .show(truncate=False)
)

+-------------+-------------+-------------+------------+
|forecast_date|temperature_c|wind_speed_ms|precip_1h_mm|
+-------------+-------------+-------------+------------+
|2026-10-12   |11.2         |2.2          |NULL        |
|2026-10-13   |10.5         |3.8          |NULL        |
|2026-10-14   |11.8         |1.8          |NULL        |
|2026-10-15   |12.5         |2.0          |NULL        |
|2026-10-16   |12.0         |3.1          |NULL        |
|2026-10-17   |10.9         |2.1          |NULL        |
|2026-10-18   |11.1         |2.8          |NULL        |
|2026-10-19   |11.3         |2.6          |NULL        |
|2026-10-20   |11.4         |3.2          |1.0         |
+-------------+-------------+-------------+------------+

